# Sequential Employee Onboarding Job

| Field | Value |
| --- | --- |
| Scenario id | `sequential-employee-onboarding` |
| Pattern | `sequential` |
| API | `Invocations API` |
| Recommended max tokens | `1000` per agent turn |

**Learning goal:** Learn how an invocation can move a structured onboarding job through required enterprise departments, where each stage consumes a concrete artifact from the stage before it.

> Use Invocations plus sequential orchestration for HRIS, ticketing, or workflow jobs where later stages depend on earlier artifacts — if departments only shared the same intake payload, concurrent orchestration would fit better.


### Runtime configuration

**Supporting code.** Reads the Ollama model and host from environment variables so the same notebook runs against any local setup without edits -- override `OLLAMA_MODEL` or `OLLAMA_HOST` before this cell to retarget it. It also creates `MCP_TOOL_FUNCTIONS`, the shared registry that fixture cells populate and `make_agent` later reads to grant tools by name. Nothing here touches the Agent Framework; this is the notebook's runtime dial.


In [ ]:
import os

OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma4:12b")
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")

# Domain tools register themselves here; every agent looks up its granted
# tools by name, so this registry is the one piece of shared runtime state.
MCP_TOOL_FUNCTIONS: dict[str, object] = {}

print(f"Ollama target: {OLLAMA_HOST} / {OLLAMA_MODEL}")


### Notebook styling

**Supporting code.** Injects the Aptos-inspired CSS the rendering helpers rely on: roster cards, tool chips, and the per-agent accent bar that colors each transcript turn. `agent_color` hashes an agent's name to a stable palette entry, which is why the same agent keeps the same color across every cell and every run. Pure presentation -- no Agent Framework surface here.


In [ ]:
from IPython.display import HTML, Markdown, display


_AGENT_COLORS = ("#3868c8", "#b0530f", "#2f7d4f", "#7d3f98", "#a3374b", "#0f7d8a", "#8a6d0f", "#54596b")

_APTOS_STYLE = """
<style>
:root { --jp-content-font-family: 'Aptos', 'Segoe UI', 'Helvetica Neue', sans-serif; }
.jp-RenderedHTMLCommon, .jp-RenderedMarkdown, .rendered_html, .jp-OutputArea-output {
    font-family: 'Aptos', 'Segoe UI', 'Helvetica Neue', sans-serif;
    line-height: 1.55;
}
.jp-RenderedHTMLCommon h1, .jp-RenderedHTMLCommon h2, .jp-RenderedHTMLCommon h3 {
    font-family: 'Aptos Display', 'Aptos', 'Segoe UI', sans-serif;
    font-weight: 600;
}
.maf-callout {
    border-left: 4px solid #3868c8; border-radius: 6px; padding: 0.6em 0.9em;
    margin: 0.6em 0; background: rgba(56, 104, 200, 0.08);
}
.maf-roster { display: flex; flex-wrap: wrap; gap: 0.6em; margin: 0.4em 0; }
.maf-card {
    border: 1px solid rgba(128, 128, 128, 0.35); border-radius: 8px;
    padding: 0.55em 0.8em; min-width: 14em; max-width: 24em; flex: 1;
}
.maf-card b { display: block; margin-bottom: 0.15em; }
.maf-card small { opacity: 0.75; }
.maf-chip {
    display: inline-block; border-radius: 999px; padding: 0 0.6em; margin: 0.2em 0.2em 0 0;
    font-size: 0.78em; border: 1px solid rgba(128, 128, 128, 0.4);
}
.maf-turn {
    border-left: 4px solid var(--maf-agent, #54596b); border-radius: 6px;
    padding: 0.45em 0.8em; margin: 0.45em 0; background: rgba(128, 128, 128, 0.07);
    white-space: pre-wrap;
}
.maf-turn b { color: var(--maf-agent, inherit); }
.maf-turn-label {
    border-left: 4px solid var(--maf-agent, #54596b); border-radius: 6px;
    padding: 0.3em 0.7em; margin: 0.7em 0 0.15em; background: rgba(128, 128, 128, 0.09);
}
.maf-turn-label b { color: var(--maf-agent, inherit); }
</style>
"""


def apply_notebook_style() -> str:
    display(HTML(_APTOS_STYLE))
    return _APTOS_STYLE


apply_notebook_style()


### Rendering helpers

**Supporting code.** `render_roster` draws one accent-colored card per agent listing its granted tools, and `render_transcript` splits workflow output on `[AgentName]` turn labels, rendering each turn's body as markdown beneath a colored label bar. This is what turns raw multi-agent output into the readable, color-coded conversation you see after the live run. Glue for the notebook, not framework API.


In [ ]:
import re as _re


def _escape_html(value) -> str:
    import html as _html

    return _html.escape(str(value))


def agent_color(name: str) -> str:
    """Deterministic per-agent accent color, stable across cells and runs."""

    return _AGENT_COLORS[sum(ord(ch) for ch in name) % len(_AGENT_COLORS)]


def render_callout(text: str) -> None:
    display(HTML("<div class='maf-callout'>" + _escape_html(text) + "</div>"))


def render_roster(scenario) -> None:
    """Render the agent roster as color-accented cards with tool chips."""

    cards = []
    for spec in scenario.agents:
        chips = "".join(
            "<span class='maf-chip'>" + _escape_html(tool) + "</span>" for tool in spec.mcp_tools
        ) or "<span class='maf-chip'>instructions only</span>"
        cards.append(
            "<div class='maf-card' style='border-top: 3px solid " + agent_color(spec.name) + "'>"
            + "<b>" + _escape_html(spec.name) + "</b>"
            + "<small>" + _escape_html(spec.description) + "</small>"
            + "<div>" + chips + "</div></div>"
        )
    display(HTML("<div class='maf-roster'>" + "".join(cards) + "</div>"))


_TURN_LABEL = _re.compile(r"^\[([A-Za-z0-9_]+)\]\s*", _re.MULTILINE)


def render_transcript(text: str) -> None:
    """Render workflow output as color-coded per-agent turns.

    Each turn's body is emitted as a ``text/markdown`` output (via
    ``Markdown``) so Jupyter renders the agent's markdown, while the
    per-agent accent color rides on an HTML label bar above the body.
    """

    pieces = _TURN_LABEL.split(text)
    preamble = pieces[0].strip()
    labeled = list(zip(pieces[1::2], pieces[2::2]))
    if not preamble and not labeled:
        display(Markdown(text))
        return
    if preamble:
        display(Markdown(preamble))
    for label, body in labeled:
        display(HTML(
            "<div class='maf-turn-label' style='--maf-agent: " + agent_color(label) + "'>"
            + "<b>" + _escape_html(label) + "</b></div>"
        ))
        display(Markdown(body.strip()))


## Pattern: Sequential

Sequential orchestration is a fixed pipeline. Each agent receives the original request plus the work accumulated so far, so the output should read like a controlled handoff from stage to stage.

Best fit: repeatable workflows where every request needs the same ordered checks.

## API Shape

**Invocations API -- per-request job payload shape.** Each request body carries `scenario`, `pattern`, `task`, `artifacts`, and `constraints`. The caller controls which orchestration runs per invocation. This fits webhooks, CI pipelines, schedulers, and service-to-service calls where the task definition changes with every request.

This is a starter scenario. The domain is intentionally lightweight so the orchestration mechanics are easy to trace before the enterprise and quote-to-cash notebooks layer in MCP tool calls and richer context.

## Pattern Anatomy

| Dimension | Detail |
| --- | --- |
| Control flow | Agents run in a fixed order, with each stage receiving the prior stage response. |
| Coordination | The graph defines the chain. The model does not decide which agent acts next. |
| Output | The terminal output includes the stage transcript. |
| Best when | Use for repeatable pipelines where every request needs the same stages. |

## Instruction-Led LLM Agents

| Agent | Role | Capabilities |
| --- | --- | --- |
| `HrCoordinatorAgent` | Produces the role profile every downstream stage consumes. | Instructions only |
| `ItProvisioningAgent` | Provisions against HR's role profile. | Instructions only |
| `SecurityComplianceAgent` | Reviews IT's proposed access list. | Instructions only |
| `PayrollBenefitsAgent` | Sets up payroll from the role profile and approved plan. | Instructions only |
| `EnablementManagerAgent` | Turns departmental artifacts into the final execution plan. | Instructions only |

> **Instructor note:** Each row is an LLM-backed agent with role instructions.
> Most agents rely on instructions alone; enterprise and quote-to-cash agents may
> also call domain tools for grounded context.


## How Sequential Plays Out in This Scenario

HR produces the role profile; IT provisions against that profile; security reviews what IT provisioned; payroll enrolls against the confirmed profile; enablement builds the first-week plan from all of it. Every stage consumes a concrete artifact from the stage before, which is exactly what this pattern is for.

In this notebook the stages run `HrCoordinatorAgent` -> `ItProvisioningAgent` -> `SecurityComplianceAgent` -> `PayrollBenefitsAgent` -> `EnablementManagerAgent`.

## Pattern Comparison

| Pattern | Control flow | Coordination cost | Latency and cost | Fails when | Choose it when |
| --- | --- | --- | --- | --- | --- |
| **sequential (this notebook)** | Fixed pipeline; each stage consumes the previous stage's output | None at runtime -- the graph is the plan | Slowest wall-clock for independent work; easiest to debug and audit | A stage needs information only a later stage produces | The order is mandated: compliance gates, artifact chains, staged approvals |
| concurrent | One fan-out to parallel lanes, one code-defined fan-in | None between lanes; the fan-in labels and combines | Best wall-clock when lanes are truly independent | Lanes secretly depend on each other's findings | Independent expert reviews of the same input, under time pressure |
| handoff | Triage names one owner; a router validates the choice | One routing decision, code-checked against allowed routes | Cheapest -- only the owner (plus an optional finisher) runs | The case genuinely needs several specialists to collaborate | Ownership depends on case facts and most specialists should not run |
| group chat | Round-robin turns in a shared transcript; a closer ends each cycle | High -- every turn rereads the whole discussion | Slow and token-hungry; the transcript itself is the product | Participants never actually react to each other | Deliberation and critique must be visible in a recorded decision |
| magentic | A manager plans, delegates, observes a ledger, and replans | Highest -- planning, delegation, and replan loops | Most expensive and least predictable run shape | The task was really a known pipeline all along | Ambiguous work where the plan must change as evidence arrives |

> **Which pattern would we actually choose?** Sequential is correct because of the artifact chain -- IT cannot provision before HR defines the role, and security reviews IT's output, not the intake form. The honest caveat lives in the scenario's own when-to-use note: if your departments only shared the intake payload and produced independent checklists, concurrent would finish faster.


### Scenario schema

**Supporting code.** Plain frozen dataclasses -- `AgentSpec` and `ScenarioSpec` -- that mirror the embedded scenario JSON: identity, pattern, roster, token budget, and the pattern- specific fields (`handoff_finisher`, `concurrent_synthesizer`, `termination_phrases`). They are deliberately not framework types: keeping the scenario contract in plain data is what lets the same spec drive five different orchestrations and both API shapes.


In [ ]:
from dataclasses import dataclass
from typing import Any, Sequence


@dataclass(frozen=True)
class AgentSpec:
    name: str
    description: str
    instructions: str
    mcp_tools: tuple[str, ...] = ()
    mcp_server: str = "enterprise_context"
    route_keywords: tuple[str, ...] = ()
    a2a_url: str | None = None


@dataclass(frozen=True)
class ScenarioSpec:
    id: str
    pattern: str
    title: str
    learning_goal: str
    when_to_use: str
    sample_task: str
    agents: tuple[AgentSpec, ...]
    max_tokens: int
    handoff_finisher: str | None = None
    concurrent_synthesizer: str | None = None
    termination_phrases: tuple[str, ...] = ()


### Load the scenario

**Supporting code.** Hydrates the embedded JSON into the `SCENARIO` object every later cell reads -- the roster the agent factory builds from, the sample prompt the live run sends, and the budget the config uses. Change a field here (an instruction, a route keyword, the budget) and rerun the downstream cells to see how behavior shifts. Data plumbing, no Agent Framework surface.


In [ ]:
import json


SCENARIO_DATA = json.loads(
    r"""
{
  "id": "sequential-employee-onboarding",
  "pattern": "sequential",
  "title": "Sequential Employee Onboarding Job",
  "learning_goal": "Learn how an invocation can move a structured onboarding job through required enterprise departments, where each stage consumes a concrete artifact from the stage before it.",
  "when_to_use": "Use Invocations plus sequential orchestration for HRIS, ticketing, or workflow jobs where later stages depend on earlier artifacts \u2014 if departments only shared the same intake payload, concurrent orchestration would fit better.",
  "max_tokens": 1000,
  "sample_task": "Build an onboarding execution plan for a new regional sales director starting in ten business days. Details: remote hire, needs CRM and billing-system access, the background check is still pending, payroll spans two states, and the manager wants a first-week enablement plan.",
  "handoff_finisher": null,
  "concurrent_synthesizer": null,
  "termination_phrases": [],
  "agents": [
    {
      "name": "HrCoordinatorAgent",
      "description": "Produces the role profile every downstream stage consumes.",
      "instructions": "Create the onboarding timeline and a concrete role profile: role, level, department, manager, start date, location, and employment type. Downstream departments provision against this profile, so state it explicitly.",
      "mcp_tools": [],
      "mcp_server": "enterprise_context",
      "route_keywords": [],
      "a2a_url": null
    },
    {
      "name": "ItProvisioningAgent",
      "description": "Provisions against HR's role profile.",
      "instructions": "Using the role profile and start date from the HR stage, list identity, laptop, email, CRM, collaboration, and device-management tasks with target dates, and output the proposed access list for security review.",
      "mcp_tools": [],
      "mcp_server": "enterprise_context",
      "route_keywords": [],
      "a2a_url": null
    },
    {
      "name": "SecurityComplianceAgent",
      "description": "Reviews IT's proposed access list.",
      "instructions": "Review the access list the IT stage proposed against least-privilege for the stated role. Approve, trim, or flag each item, and add mandatory training, data handling, and audit evidence requirements.",
      "mcp_tools": [],
      "mcp_server": "enterprise_context",
      "route_keywords": [],
      "a2a_url": null
    },
    {
      "name": "PayrollBenefitsAgent",
      "description": "Sets up payroll from the role profile and approved plan.",
      "instructions": "Using the role profile's location, employment type, and start date, add payroll, benefits, tax, direct deposit, and regional compliance actions consistent with the approved access plan.",
      "mcp_tools": [],
      "mcp_server": "enterprise_context",
      "route_keywords": [],
      "a2a_url": null
    },
    {
      "name": "EnablementManagerAgent",
      "description": "Turns departmental artifacts into the final execution plan.",
      "instructions": "Combine the role profile, provisioning tasks, security-approved access plan, and payroll actions into a concise checklist with dates, blockers, and first-week success measures.",
      "mcp_tools": [],
      "mcp_server": "enterprise_context",
      "route_keywords": [],
      "a2a_url": null
    }
  ]
}
    """
)
AGENTS = tuple(
    AgentSpec(
        name=item["name"],
        description=item["description"],
        instructions=item["instructions"],
        mcp_tools=tuple(item.get("mcp_tools", [])),
        mcp_server=item.get("mcp_server", "enterprise_context"),
        route_keywords=tuple(item.get("route_keywords", [])),
        a2a_url=item.get("a2a_url"),
    )
    for item in SCENARIO_DATA["agents"]
)
SCENARIO = ScenarioSpec(
    id=SCENARIO_DATA["id"],
    pattern=SCENARIO_DATA["pattern"],
    title=SCENARIO_DATA["title"],
    learning_goal=SCENARIO_DATA["learning_goal"],
    when_to_use=SCENARIO_DATA["when_to_use"],
    sample_task=SCENARIO_DATA["sample_task"],
    agents=AGENTS,
    max_tokens=SCENARIO_DATA["max_tokens"],
    handoff_finisher=SCENARIO_DATA.get("handoff_finisher"),
    concurrent_synthesizer=SCENARIO_DATA.get("concurrent_synthesizer"),
    termination_phrases=tuple(SCENARIO_DATA.get("termination_phrases", [])),
)

print(f"Loaded {SCENARIO.title} with {len(SCENARIO.agents)} agents.")


### Inspection helpers

**Supporting code.** `agent_capability_map` summarizes who can do what, `mcp_tool_context` reports which domain tools exist, and `tools_for_agent` resolves an agent's granted tool names to the actual callables `make_agent` will attach. Inspecting the roster this way before running is a habit worth keeping: most orchestration surprises trace back to an agent having more, fewer, or different tools than you assumed.


In [ ]:
def tools_for_agent(spec: AgentSpec) -> list[object]:
    tools: list[object] = []
    for tool_name in spec.mcp_tools:
        try:
            tools.append(MCP_TOOL_FUNCTIONS[tool_name])
        except KeyError as exc:
            raise ValueError(f"Missing inlined tool '{tool_name}' for {spec.name}.") from exc
    return tools


def scenario_summary(scenario: ScenarioSpec) -> dict[str, str]:
    return {
        "id": scenario.id,
        "title": scenario.title,
        "pattern": scenario.pattern,
        "learning_goal": scenario.learning_goal,
        "when_to_use": scenario.when_to_use,
        "max_tokens": str(scenario.max_tokens),
        "sample": getattr(scenario, "sample_task"),
    }


def agent_capability_map(scenario: ScenarioSpec) -> list[dict[str, Any]]:
    return [
        {
            "agent": spec.name,
            "description": spec.description,
            "instructions": spec.instructions,
            "mcp_tools": list(spec.mcp_tools),
            "mcp_server": spec.mcp_server if spec.mcp_tools else None,
        }
        for spec in scenario.agents
    ]


def mcp_tool_context(scenario: ScenarioSpec) -> dict[str, Any]:
    tools_by_agent = {spec.name: list(spec.mcp_tools) for spec in scenario.agents if spec.mcp_tools}
    all_tools_used = sorted({tool for spec in scenario.agents for tool in spec.mcp_tools})
    return {
        "uses_mcp": bool(all_tools_used),
        "tools_by_agent": tools_by_agent,
        "all_tools_used": all_tools_used,
    }
def build_invocation_prompt(payload: dict[str, object]) -> str:
    artifacts = "\n".join(f"- {item}" for item in payload.get("artifacts", [])) or "- No artifacts supplied."
    constraints = "\n".join(f"- {item}" for item in payload.get("constraints", [])) or "- No explicit constraints."
    return (
        f"Scenario: {payload['scenario']} - {SCENARIO.title}\n"
        f"Pattern: {payload['pattern']}\n"
        f"Learning goal: {SCENARIO.learning_goal}\n"
        f"Subject: {payload['subject']}\n"
        f"Task: {payload['task']}\n\n"
        f"Artifacts:\n{artifacts}\n\n"
        f"Constraints:\n{constraints}\n\n"
        "Session context:\nNo prior turns for this session.\n\n"
        "Return actionable findings. Do not claim to have inspected artifacts beyond the provided names and context."
    )


### Sample prompt and budget

**Supporting code.** Pins the two run-defining inputs: `MAX_TOKENS`, the per-turn generation budget (this scenario's recommended value unless `OLLAMA_MAX_TOKENS` overrides it), and `SAMPLE_PROMPT`, the exact text the live run will dispatch. It then renders the roster so you can see the team -- and each agent's accent color -- before any orchestration happens. Budgets matter locally: too low truncates an agent mid- thought, too high slows every turn.


In [ ]:
import json


MAX_TOKENS = int(os.getenv("OLLAMA_MAX_TOKENS", str(SCENARIO.max_tokens)))
INVOCATION_PAYLOAD = {
    "scenario": SCENARIO.id,
    "pattern": SCENARIO.pattern,
    "task": SCENARIO.sample_task,
    "subject": "new-hire: regional sales director",
    "artifacts": [
        "hris/new-hire-profile.json",
        "access-templates/sales-director.yaml",
        "enablement/first-week-plan.md",
    ],
    "constraints": [
        "Include HR, IT, security, payroll, and manager enablement actions.",
        "Return owners, timing, blockers, and first-week success measures.",
    ],
    "stream": False,
}
SAMPLE_PROMPT = build_invocation_prompt(INVOCATION_PAYLOAD)

render_roster(SCENARIO)
print(json.dumps(scenario_summary(SCENARIO), indent=2))
print(json.dumps(agent_capability_map(SCENARIO), indent=2))
if mcp_tool_context(SCENARIO)["uses_mcp"]:
    print(json.dumps(mcp_tool_context(SCENARIO), indent=2))


### Ollama configuration

**Supporting code.** A frozen `OllamaAgentConfig` dataclass captures everything one agent's chat client needs -- model, host, temperature, context window, the scenario's token budget, keep-alive, and the think flag -- with environment variables as the override channel. Freezing it guarantees every agent in a run shares identical runtime settings. Local-runtime plumbing, independent of any Agent Framework class.


In [ ]:
from dataclasses import dataclass
from typing import Any

from agent_framework.ollama import OllamaChatClient


DEFAULT_OLLAMA_TEMPERATURE = 0.0
DEFAULT_OLLAMA_NUM_CTX = 8192
DEFAULT_OLLAMA_KEEP_ALIVE = "10m"
DEFAULT_OLLAMA_THINK = False

# Ollama's chat endpoint rejects a few OpenAI-style options; we strip these later.
_UNSUPPORTED_OLLAMA_CHAT_OPTIONS = {
    "allow_multiple_tool_calls",
    "conversation_id",
    "logit_bias",
    "metadata",
    "store",
    "user",
}


@dataclass(frozen=True)
class OllamaAgentConfig:
    model: str
    host: str
    temperature: float
    num_ctx: int
    max_tokens: int
    keep_alive: str
    think: bool

    def default_options(self) -> dict[str, Any]:
        return {
            "temperature": self.temperature,
            "num_ctx": self.num_ctx,
            "max_tokens": self.max_tokens,
            "keep_alive": self.keep_alive,
            "think": self.think,
        }


def parse_env_bool(name: str, default: bool) -> bool:
    value = os.getenv(name)
    if value is None or value.strip() == "":
        return default
    normalized = value.strip().lower()
    if normalized in {"1", "true", "yes", "on"}:
        return True
    if normalized in {"0", "false", "no", "off"}:
        return False
    raise ValueError(f"{name} must be true or false.")


def build_ollama_config(
    *,
    model: str | None = None,
    host: str | None = None,
    temperature: float | None = None,
    num_ctx: int | None = None,
    max_tokens: int | None = None,
    keep_alive: str | None = None,
    think: bool | None = None,
) -> OllamaAgentConfig:
    return OllamaAgentConfig(
        model=model or os.getenv("OLLAMA_MODEL") or "gemma4:12b",
        host=host or os.getenv("OLLAMA_HOST") or "http://localhost:11434",
        temperature=temperature
        if temperature is not None
        else float(os.getenv("OLLAMA_TEMPERATURE", str(DEFAULT_OLLAMA_TEMPERATURE))),
        num_ctx=num_ctx if num_ctx is not None else int(os.getenv("OLLAMA_NUM_CTX", str(DEFAULT_OLLAMA_NUM_CTX))),
        max_tokens=max_tokens if max_tokens is not None else int(os.getenv("OLLAMA_MAX_TOKENS", "1000")),
        keep_alive=keep_alive or os.getenv("OLLAMA_KEEP_ALIVE") or DEFAULT_OLLAMA_KEEP_ALIVE,
        think=think if think is not None else parse_env_bool("OLLAMA_THINK", DEFAULT_OLLAMA_THINK),
    )


### Chat-client shim

**Supporting code.** A thin `OllamaChatClient` subclass that strips chat options the local Ollama server would reject before each request goes out. Adapters like this are common at the edge between a framework and a specific model server: the framework speaks a superset, the server accepts a subset, and the shim reconciles them without touching any agent code.


In [ ]:
class ScenarioOllamaChatClient(OllamaChatClient):
    """OllamaChatClient that drops chat options the local Ollama server rejects."""

    def _prepare_options(self, messages: Any, options: Any) -> dict[str, Any]:
        filtered = {
            key: value
            for key, value in dict(options).items()
            if key not in _UNSUPPORTED_OLLAMA_CHAT_OPTIONS
        }
        return super()._prepare_options(messages, filtered)


### make_agent

**Agent Framework primitive.** The factory this whole repo pivots on: `client.as_agent(...)` combines a chat client, the spec's role instructions (prefixed with the agent's name), and any granted tools into a runnable agent -- or returns an `A2AAgent` when the spec points at a remote peer. Every orchestration pattern downstream consumes agents built exactly here, which is why instructions and tool grants live in the scenario spec rather than in pattern code. This is the Agent Framework's core agent-construction call.


In [ ]:
def make_agent(spec: Any, *, config: OllamaAgentConfig | None = None) -> Any:
    if spec.a2a_url:
        from agent_framework.a2a import A2AAgent

        url = spec.a2a_url
        if not url.startswith("http"):
            url = os.getenv("A2A_PARTNER_BASE_URL", "http://localhost:8765").rstrip("/") + url
        return A2AAgent(name=spec.name, description=spec.description, url=url)

    resolved = config or build_ollama_config()
    instructions = f"You are {spec.name}. {spec.instructions}"
    tools = tools_for_agent(spec)
    return ScenarioOllamaChatClient(host=resolved.host, model=resolved.model).as_agent(
        name=spec.name,
        description=spec.description,
        instructions=instructions,
        tools=tools or None,
        default_options=resolved.default_options(),
        require_per_service_call_history_persistence=True,
    )


print("Agent factory ready: make_agent(spec) creates an instruction-led Ollama agent "
      "with its granted tools attached.")


### Framework imports and message helpers

**Agent Framework primitive.** Imports the workflow surface every pattern in this repo builds on -- `Executor`, `WorkflowBuilder`, `WorkflowContext`, `AgentExecutor`, and the `@handler` decorator -- plus `make_request` and `response_text`, two helpers that wrap plain text into an `AgentExecutorRequest` and pull it back out of a response. Messages are the typed boundary between workflow nodes: everything an agent receives or returns passes through these shapes.


In [ ]:
import re
from typing import Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    handler,
)


# State keys shared across executors: the running transcript, and the stopwords
# the handoff router strips when it derives routing keywords from agent names.
_TRANSCRIPT_KEY = "transcript"
_STOPWORDS = {"agent", "specialist", "the", "and", "for", "with", "that", "from", "into"}


def make_request(text: str) -> AgentExecutorRequest:
    return AgentExecutorRequest(messages=[Message(role="user", contents=[text])])


def response_text(response: AgentExecutorResponse) -> str:
    agent_response = getattr(response, "agent_response", None)
    return (getattr(agent_response, "text", None) or "").strip()


### Transcript state

**Supporting code.** A helper that appends a `[AgentName] text` line to the shared transcript stored in workflow state via `ctx.get_state`/`ctx.set_state`. Keeping the transcript in state -- rather than inside any single executor -- is what lets gates, routers, and output executors all read the same running history. Bookkeeping the executors reuse, not framework API itself.


In [ ]:
def _append_transcript(ctx: WorkflowContext[Any], author: str, text: str) -> list[str]:
    transcript = list(ctx.get_state(_TRANSCRIPT_KEY) or [])
    transcript.append(f"[{author}] {text}")
    ctx.set_state(_TRANSCRIPT_KEY, transcript)
    return transcript


### Your first executor

**Agent Framework primitive.** `PromptDispatchExecutor` is the minimal custom executor: subclass `Executor`, mark an async method with `@handler`, and the framework routes matching messages to it. This one seeds the prompt and an empty transcript into state, then `send_message`s the first request -- the entry node of every graph in this repo. The handler signature (input type plus `WorkflowContext[OutputType]`) is how the framework knows what a node consumes and emits.


In [ ]:
class PromptDispatchExecutor(Executor):
    @handler
    async def dispatch(self, prompt: str, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        ctx.set_state("prompt", prompt)
        ctx.set_state(_TRANSCRIPT_KEY, [])
        await ctx.send_message(make_request(prompt))


### Agents as workflow nodes

**Agent Framework primitive.** `_agent_executor` wraps a factory-built agent in an `AgentExecutor`, giving it a graph id and making it addressable as a workflow node -- the bridge between the agent world (instructions, tools, chat client) and the workflow world (edges, handlers, state). The slugified id matters more than it looks: the handoff router targets specialists by exactly these ids.


In [ ]:
def _slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


def _agents_for(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> list[Any]:
    return [make_agent(spec, config=config) for spec in scenario.agents]


def _agent_executor(spec_index: int, scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> AgentExecutor:
    spec = scenario.agents[spec_index]
    return AgentExecutor(make_agent(spec, config=config), id=_slug(spec.name))


print("Workflow plumbing ready: dispatch executor, shared transcript state, and "
      "request/response helpers.")


### StageGateExecutor: carry the transcript forward

**Agent Framework primitive.** The sequential pattern's engine. After each stage responds, this gate appends the stage's output to the shared transcript and sends the next agent a fresh request containing the original prompt plus all accumulated work -- so stage N+1 sees everything stages 1 through N produced, not just the last message. The explicit 'do not repeat the earlier stages' instruction is a small but real guard against agents re-answering the whole task.


In [ ]:
class StageGateExecutor(Executor):
    def __init__(self, id: str, *, stage_name: str) -> None:
        super().__init__(id=id)
        self._stage_name = stage_name

    @handler
    async def gate(self, response: AgentExecutorResponse, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        transcript = _append_transcript(ctx, self._stage_name, response_text(response))
        prompt = ctx.get_state("prompt") or ""
        carried = "\n".join(transcript)
        await ctx.send_message(
            make_request(
                f"Original request:\n{prompt}\n\nWork so far:\n{carried}\n\n"
                "Add your stage's contribution; do not repeat the earlier stages."
            )
        )


### SequentialOutputExecutor: yield the final transcript

**Agent Framework primitive.** The terminal node: instead of forwarding another request, its handler calls `ctx.yield_output(...)` with the joined transcript, and that value becomes the workflow's result. Every pattern in this repo ends at an executor that yields rather than sends -- yielding is how a graph declares 'this is the answer'.


In [ ]:
class SequentialOutputExecutor(Executor):
    def __init__(self, id: str, *, stage_name: str) -> None:
        super().__init__(id=id)
        self._stage_name = stage_name

    @handler
    async def finish(self, response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
        transcript = _append_transcript(ctx, self._stage_name, response_text(response))
        await ctx.yield_output("\n\n".join(transcript))


### Preview the stage handoff

**Supporting code.** An offline sanity check -- no model call -- that prints the exact prompt one stage gate hands the next, using stand-in findings. Read it once before the live run: when a later stage misbehaves, the first question is always 'what did it actually receive?', and this cell shows you precisely that shape.


In [ ]:
# Demo (offline): the exact prompt a stage gate hands to the next stage.
_demo_transcript = [
    f"[{SCENARIO.agents[0].name}] First-stage findings would appear here.",
    f"[{SCENARIO.agents[1].name}] Second-stage findings build on them.",
]
print("Original request:\n" + SAMPLE_PROMPT + "\n\nWork so far:\n" + "\n".join(_demo_transcript))


### Wire the graph with WorkflowBuilder

**Agent Framework primitive.** `WorkflowBuilder` assembles the executors into a fixed graph: `start_executor` and `output_from` declare the ends, and `add_edge` chains dispatch -> agent -> gate -> agent ... -> output, weaving a `StageGateExecutor` between consecutive agents. Read the loop and notice what is absent -- no model input, no branching. The topology is decided entirely in code, which is the sequential pattern's promise.


In [ ]:
def build_sequential_workflow(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> Any:
    agents = [_agent_executor(i, scenario, config=config) for i in range(len(scenario.agents))]
    dispatch = PromptDispatchExecutor(id="dispatch")
    output = SequentialOutputExecutor(id="final_output", stage_name=scenario.agents[-1].name)
    builder = WorkflowBuilder(start_executor=dispatch, output_from=[output])
    builder.add_edge(dispatch, agents[0])
    for index in range(len(agents) - 1):
        gate = StageGateExecutor(id=f"gate_{index}", stage_name=scenario.agents[index].name)
        builder.add_edge(agents[index], gate)
        builder.add_edge(gate, agents[index + 1])
    builder.add_edge(agents[-1], output)
    return builder.build()


### Compile and build

**Supporting code.** `build_workflow` resolves the Ollama config (model, host, and this scenario's token budget) and hands it to the builder above; `build()` then compiles the executor graph into a runnable workflow object. The wrapper is notebook glue so later cells can rebuild with overrides -- try `build_workflow(max_tokens=250)` for a faster smoke run -- while `build()` is the actual framework call. The printed type confirms what the framework produced.


In [ ]:
def build_workflow(
    scenario: ScenarioSpec = SCENARIO,
    *,
    max_tokens: int | None = None,
    **config_overrides: Any,
) -> Any:
    config = build_ollama_config(max_tokens=max_tokens or MAX_TOKENS, **config_overrides)
    return build_sequential_workflow(scenario, config=config)


workflow = build_workflow(SCENARIO, max_tokens=MAX_TOKENS)
print(
    f"Built the {SCENARIO.pattern} workflow over {len(SCENARIO.agents)} agents: "
    + type(workflow).__name__
)


## Flow Diagram

The diagram below shows a linear chain of 5 stages with a stage-gate executor between each pair attached to the Invocations API /invocations.
Solid arrows are orchestration edges. Dashed arrows (`-.->`) are tool calls.
Domain tool nodes use a stadium shape.


In [ ]:
import base64
import html
from dataclasses import dataclass

from IPython.display import HTML, display


@dataclass(frozen=True)
class ScenarioFlowDiagram:
    title: str
    mermaid: str
    image_url: str


def scenario_flow_diagram(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    mermaid = _sequential_diagram(scenario, api_boundary="Invocations API /invocations", input_label="Job payload")
    return ScenarioFlowDiagram(
        title=f"{scenario.title} Flow",
        mermaid=mermaid,
        image_url=_mermaid_image_url(mermaid),
    )


def display_scenario_flow(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    diagram = scenario_flow_diagram(scenario)
    display(
        HTML(
            '<figure style="margin: 0">'
            f'<img src="{html.escape(diagram.image_url)}" alt="{html.escape(diagram.title)}" '
            'style="max-width: 100%; height: auto;" />'
            f'<figcaption style="font-size: 0.9em; color: #555;">{html.escape(diagram.title)}</figcaption>'
            "</figure>"
        )
    )
    return diagram



def _sequential_diagram(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> str:
    lines = _header(scenario, api_boundary=api_boundary, input_label=input_label)
    previous = "orchestrator"
    pairs: list[tuple[AgentSpec, str]] = []
    for index, agent in enumerate(scenario.agents, start=1):
        node = f"agent{index}"
        lines.append(f"    {previous} -->|stage {index}| {node}[{_label(agent.name)}]")
        previous = node
        pairs.append((agent, node))
    lines.append(f"    {previous} --> output[Invocation summary]")
    lines.extend(_mcp_tool_links(pairs))
    return "\n".join(lines)



def _header(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> list[str]:
    return [
        "%%{init: {'theme': 'neutral'}}%%",
    "flowchart TD",
        f"    client[{_label(input_label)}] --> api[{_label(api_boundary)}]",
        f"    api --> scenario[{_label('Scenario: ' + scenario.id)}]",
        f"    scenario --> orchestrator{{{_label(scenario.pattern + ' orchestration')}}}",
    ]


def _mcp_tool_links(pairs: list[tuple[AgentSpec, str]]) -> list[str]:
    lines: list[str] = []
    for agent, node_id in pairs:
        for tool in agent.mcp_tools:
            lines.append(f"    {node_id} -.->|mcp tool| tool_{tool}([{_label(tool)}])")
    return lines


def quote_to_cash_flow_diagram(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    mermaid = _quote_to_cash_source(scenario, api_boundary="Invocations API /invocations")
    return ScenarioFlowDiagram(
        title=f"{scenario.title} Quote-To-Cash Flow",
        mermaid=mermaid,
        image_url=_mermaid_image_url(mermaid),
    )


def display_quote_to_cash_flow(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    diagram = quote_to_cash_flow_diagram(scenario)
    display(
        HTML(
            '<figure style="margin: 0">'
            f'<img src="{html.escape(diagram.image_url)}" alt="{html.escape(diagram.title)}" '
            'style="max-width: 100%; height: auto;" />'
            f'<figcaption style="font-size: 0.9em; color: #555;">{html.escape(diagram.title)}</figcaption>'
            "</figure>"
        )
    )
    return diagram


def _quote_to_cash_source(scenario: ScenarioSpec, *, api_boundary: str) -> str:
    names = {agent.name for agent in scenario.agents}

    def node(role: str) -> str:
        return role if role in names else next(iter(names))

    lines = [
        "%%{init: {'theme': 'neutral'}}%%",
    "flowchart TD",
        f"    client[{_label('Quote request begins in CRM')}] --> api[{_label(api_boundary)}]",
        f"    api --> scenario[{_label('Scenario: ' + scenario.id)}]",
        f"    scenario --> orchestrator{{{_label(scenario.pattern + ' orchestration')}}}",
        f"    orchestrator --> crm[{_label('CRM system')}]",
        f"    crm -->|wave 1| trigger[{_label(node('QuoteTriggerAgent'))}]",
        f"    crm -->|wave 1| customer[{_label(node('CustomerContextAgent'))}]",
        f"    trigger --> store1[({_label('Orchestration store: customer context')})]",
        "    customer --> store1",
        f"    store1 -. {_label('deallocate wave 1')} .-> freed1(({_label('agents released')}))",
        f"    store1 --> product[{_label('Product / SKU system')}]",
        f"    product -->|wave 2| sku[{_label(node('SkuDiscoveryAgent'))}]",
        f"    product -->|wave 2| fit[{_label(node('ProductFitAgent'))}]",
        f"    sku --> store2[({_label('Orchestration store: product context')})]",
        "    fit --> store2",
        f"    store2 -. {_label('deallocate wave 2')} .-> freed2(({_label('agents released')}))",
        f"    store2 --> pricingsys[{_label('Pricing / finance / legal system')}]",
        f"    pricingsys -->|wave 3| pricing[{_label(node('PricingTermsAgent'))}]",
        f"    pricing --> generation[{_label(node('QuoteGenerationAgent'))}]",
        f"    generation --> quote[/{_label('Final quote package')}/]",
    ]
    return "\n".join(lines)


def _label(value: str) -> str:
    return value.replace('"', "'")


def _mermaid_image_url(mermaid: str) -> str:
    encoded = base64.urlsafe_b64encode(mermaid.encode("utf-8")).decode("ascii").rstrip("=")
    return f"https://mermaid.ink/img/{encoded}"


flow_diagram = display_scenario_flow(SCENARIO)
print(flow_diagram.mermaid)


### Read the run output

**Supporting code.** Utilities that unpack a finished run: `result.get_outputs()` returns the workflow's yielded outputs, and `get_intermediate_outputs()` exposes per-participant turns where the orchestration surfaces them (group chat and magentic). Everything else is string parsing that feeds `render_transcript`, so the color-coded turns you see below are exactly what the executors yielded -- those two calls are the only Agent Framework touchpoints.


In [ ]:
from collections.abc import Mapping, Sequence


def workflow_result_to_text(result: Any) -> str:
    outputs = result.get_outputs() if hasattr(result, "get_outputs") else result
    intermediate = result.get_intermediate_outputs() if hasattr(result, "get_intermediate_outputs") else []
    if not outputs:
        intermediate_text = join_readable_outputs(intermediate)
        return clean_workflow_text(intermediate_text) or "No workflow output was produced."
    output_text = join_readable_outputs(outputs)
    if intermediate and should_use_intermediate_outputs(output_text):
        intermediate_text = join_readable_outputs(intermediate)
        if intermediate_text:
            return clean_workflow_text(intermediate_text)
    return clean_workflow_text(output_text) or "No readable workflow text was produced."


def join_readable_outputs(outputs: Any) -> str:
    return "\n\n".join(text for output in outputs if (text := agent_response_to_text(output)))


def should_use_intermediate_outputs(output_text: str) -> bool:
    normalized = output_text.strip().lower()
    if not normalized:
        return True
    if len(normalized) >= 160:
        return False
    markers = (
        "termination condition",
        "maximum reset count",
        "maximum stall count",
        "workflow terminated",
        "group chat has reached its termination condition",
    )
    return any(marker in normalized for marker in markers)


def agent_response_to_text(value: Any) -> str:
    text = clean_workflow_text(extract_text(value))
    return text


def clean_workflow_text(text: str) -> str:
    """Remove leading framework status lines when useful scenario text follows."""

    lines = text.splitlines()
    while lines and is_framework_status_line(lines[0]) and any(line.strip() for line in lines[1:]):
        lines.pop(0)
        while lines and not lines[0].strip():
            lines.pop(0)
    return "\n".join(lines).strip()


def is_framework_status_line(line: str) -> bool:
    normalized = line.strip().lower()
    return (
        normalized.startswith("invalid next speaker:")
        or normalized.startswith("magentic orchestrator:")
        or normalized.startswith("maximum consecutive function call errors reached")
    )


def extract_text(value: Any, seen: set[int] | None = None) -> str:
    if value is None:
        return ""
    if seen is None:
        seen = set()
    value_id = id(value)
    if value_id in seen:
        return ""
    seen.add(value_id)
    if isinstance(value, str):
        return "" if is_object_repr(value) else value
    text = getattr(value, "text", None)
    if isinstance(text, str) and text and not is_object_repr(text):
        return text
    messages = getattr(value, "messages", None)
    if messages:
        parts: list[str] = []
        for message in messages:
            author = getattr(message, "author_name", None) or getattr(message, "role", None) or "assistant"
            message_text = extract_text(message, seen)
            if message_text:
                parts.append(f"[{author}] {message_text}")
        if parts:
            return "\n".join(parts)
    contents = getattr(value, "contents", None)
    if contents:
        return "\n".join(part for content in contents if (part := extract_text(content, seen)))
    items = getattr(value, "items", None)
    if items and not callable(items):
        return "\n".join(part for item in items if (part := extract_text(item, seen)))
    result = getattr(value, "result", None)
    if result is not None:
        return extract_text(result, seen)
    if isinstance(value, Mapping):
        parts = [extract_text(value.get(key), seen) for key in ("text", "content", "message", "summary", "result")]
        return "\n".join(part for part in parts if part)
    if isinstance(value, Sequence) and not isinstance(value, (bytes, bytearray, str)):
        return "\n".join(part for item in value if (part := extract_text(item, seen)))
    fallback = str(value)
    return "" if is_object_repr(fallback) else fallback


def is_object_repr(value: str) -> bool:
    return value.startswith("<") and " object at 0x" in value and value.endswith(">")




print("Result formatting ready: workflow_result_to_text(...) turns framework events "
      "into readable text.")


## Live Run

Each agent output is captured by a `StageGateExecutor` and appended to a growing transcript. The next agent receives both the original prompt and the accumulated work so far. The final cell prints the complete stage-by-stage log.

> **Instructor note:** `gemma4:12b` runs with `think: False` by default in this repo.
> Set `OLLAMA_THINK=true` before the environment cell to enable chain-of-thought reasoning --
> useful when debugging unexpected routing decisions or tool call sequences.


In [ ]:
import io
from contextlib import redirect_stderr, redirect_stdout


workflow = build_workflow(SCENARIO, max_tokens=MAX_TOKENS)
_framework_logs = io.StringIO()
with redirect_stdout(_framework_logs), redirect_stderr(_framework_logs):
    result = await workflow.run(SAMPLE_PROMPT)
framework_logs = _framework_logs.getvalue()
output_text = workflow_result_to_text(result)
if SCENARIO.pattern == "group-chat" and should_use_intermediate_outputs(output_text):
    output_text = group_chat_learning_summary(SCENARIO, SAMPLE_PROMPT, output_text)

if not output_text.strip():
    raise RuntimeError("Workflow completed but produced no readable text.")

render_transcript(output_text)


## What to Inspect

Compare the first stage output to the final editor output. Later stages should build on prior work, not repeat it -- each `StageGateExecutor` carries the full transcript forward. If a stage ignores prior context, inspect its instructions and the gate prompt to see exactly what the agent received.

> **Scenario spotlight:** Each stage consumes an artifact: role profile -> proposed access list -> security-trimmed plan -> payroll actions. Check the chain survives intact.


## Experiments

- Change the role to a contractor in the payload and watch which downstream stages adapt.
- Change `INVOCATION_PAYLOAD['task']`, `subject`, `artifacts`, or `constraints` and rerun the live cell.
- Override `OLLAMA_MODEL` or `OLLAMA_HOST` before the environment cell to target a different local Ollama setup.
- Inspect `agent_capability_map(SCENARIO)` and tighten one agent's instructions to see how orchestration behavior changes.
- Lower `MAX_TOKENS` for faster smoke tests or raise it when sequential needs more room.
